In [ ]:
# Standard libraries
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from functools import partial
from itertools import product

# Specific libraries
import nibabel as nib
import seaborn as sns
import plotly.express as px

# Import custom modules
import pp_utils as pp

In [ ]:
# Define directories
CALIB_DIR = Path("./snr")
SCAN_DIR = CALIB_DIR / "scan_data"
TEMP_DIR = CALIB_DIR / "temperature_data"
OUTPUT_DIR = CALIB_DIR / "derivatives"
testname = CALIB_DIR.stem

if not CALIB_DIR.is_dir():
    raise NotADirectoryError(f"Expected a directory, got: {CALIB_DIR}") 
if not SCAN_DIR.is_dir():
    raise NotADirectoryError(f"{CALIB_DIR.stem} doesn't contain subdirectory named {SCAN_DIR.stem}") 
if not TEMP_DIR.is_dir():
    raise NotADirectoryError(f"{CALIB_DIR.stem} doesn't contain subdirectory named {TEMP_DIR.stem}") 

OUTPUT_DIR.mkdir(exist_ok=True)

### LOAD & UNPACK

In [ ]:
# Load in data and save basic features 
dicom_fnames = pp.scrape_files(SCAN_DIR, filetype=["MRe", "dcm"])
dicom_fnames.sort(key = lambda x: pp.dicom_scantime(x))
print(f"The full dataset contains {len(dicom_fnames)} DICOM files")
dataset_unpacked = list(map(pp.process_dicom, dicom_fnames))
dataset_df = pd.DataFrame(dataset_unpacked)
print("Scans unpacked into pandas DataFrame structure")

### PAIR W/ TEMP DATA

In [ ]:
# Now pair scans with their temperature measurements and remove any scans without valid temperature measurements
temp_fname = TEMP_DIR / "temperatures.csv"
temp_log = np.loadtxt(temp_fname, dtype=str, delimiter=",", skiprows=1)
temp_log_dt = [[pp.convert_timestamp(datetime_string), float(temp)] for datetime_string, temp in temp_log]
temp_log_dt_np = np.array(temp_log_dt)

dataset_df["temperature"] = dataset_df["scan_time"].apply(partial(pp.estimate_temp, temp_log=temp_log_dt, time_window_minutes=90, f_past=0.5, n_min=None))
print(f"Full dataset contains {dataset_df.shape[0]} scans")

n_lacking_temp = np.count_nonzero(dataset_df["temperature"].isna())
print(f"Removing {n_lacking_temp} scans - don't have corresponding temperature data")

valid_df = dataset_df[dataset_df["temperature"].notna()].copy()
print(f"'Temperature trimmed' dataset contains {valid_df.shape[0]} scans")

fig, ax = plt.subplots(1, 1, figsize=(20,8))
ax.scatter(temp_log_dt_np[:,0], 
           temp_log_dt_np[:,1], 
           marker="x", 
           linewidth=1 , 
           c="k", 
           label="Raw temperature measurements")

ax.scatter(valid_df["scan_time"].values, 
           valid_df["temperature"].values, 
           marker=".", 
           linewidth=0.1, 
           c="r",
           label="Estimated phantom measurement during scan")
ax.legend()
ax.set_ylabel("Temperature (C)")
ax.set_xlabel("Time")
fig.tight_layout()
plt.show()

In [ ]:
def plot_num_of_repeats(df, col_name):

    num_repeats = df.groupby(col_name).size().rename("count").reset_index()

    fig, ax = plt.subplots(figsize=(16, 9))
    sns.barplot(data=num_repeats, x=col_name, y="count", hue=col_name, palette="husl", ax=ax)

    # Tidy up axes
    ax.set_xticks(np.arange(num_repeats.values.shape[0]), num_repeats[col_name])
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="center")
    ax.set_yticks(range(0, num_repeats["count"].values.max() + 1))
    ax.set_ylabel("Number of Repeats")
    fig.tight_layout()

    plt.show()

plot_num_of_repeats(valid_df, "seq_name")


### REGISTRATION

In [ ]:
# Extract images from dataframe
# mppca_image_4d = np.stack(denoised_df["mppca_image"].values, axis=-1)
image_4d = np.stack(valid_df["image"].values, axis=-1)
aff_0 = valid_df["affine"].values[0]
spacing_0 = pp.spacing_from_affine(aff_0)

# Extract b-values in order to be able to identify b=0 scan to register other scans to
bvals = valid_df["b_value"].values

# Perform registration:
# function returns a registered 4D image array and also
# a numpy array of indices for which registration failed
reg_4d, i_drop = pp.register_to_b0(data_4d=image_4d, 
                                   bvals=bvals, 
                                   spacing=spacing_0,
                                   b0_threshold=5)

# Create a new dataframe with only the measurements that were successfully
# registered and append a column for registered scans
reg_df = valid_df.drop(index=i_drop).reset_index(drop=True)
reg_4d_reshuffled = np.transpose(reg_4d, (3,0,1,2))
reg_df["reg_image"] = list(reg_4d_reshuffled)

### CREATE b=0 MASK

In [ ]:
b0_threshold = 5
b0_scans = np.stack(reg_df[reg_df["b_value"] < b0_threshold]["reg_image"].values, axis=-1)
b0_avg = np.mean(b0_scans, axis=-1)
mask = pp.get_mask(b0_avg, lt_percentile=55).astype(bool)

# Plot the resulting masked data
plotdata = b0_avg
plotdata[mask==0] = 0
zmin = np.percentile(plotdata[mask], 1)
zmax = np.percentile(plotdata[mask], 99)
px.imshow(plotdata, animation_frame=2, zmin=zmin, zmax=zmax, height=800, width=1000, title="Masked b=0 image")

## SAVE OUTPUTS

In [ ]:
raw_images = np.stack(reg_df["image"].values, axis=-1)
raw_nii = nib.Nifti1Image(raw_images, affine=aff_0)
pp.save_nifti(raw_nii, OUTPUT_DIR, testname + "_" + "RAW.nii.gz")

reg_images = np.stack(reg_df["reg_image"].values, axis=-1)
reg_nii = nib.Nifti1Image(reg_images, affine=aff_0)
pp.save_nifti(reg_nii, OUTPUT_DIR, testname + "_" + "RAW_REG.nii.gz")

b0_mask_nii = nib.Nifti1Image(mask.astype(float), aff_0)
pp.save_nifti(b0_mask_nii, OUTPUT_DIR, testname + "_" + "MASK.nii.gz")

param_df = reg_df.drop(["image", "reg_image"], axis=1, inplace=False)
param_fname = OUTPUT_DIR / (testname + "_" + "params.h5")
param_df.to_hdf(param_fname, key="df", mode="w", index=False)